In [ ]:
import os
import json
import logging
import pandas as pd
import numpy as np
from pprint import pprint
import nltk
import itertools
import re

In [ ]:

nltk.download('punkt')

section_types = ['TITLE', 'ABSTRACT', 'INTRO', 'FIG', 'METHODS', 'RESULTS']
directory = os.getcwd()
data_rows = []

# Compile regex patterns
patterns = [
    re.compile(r'\bFig\.(?=\d)'),
    re.compile(r'\bet al\.'),
    re.compile(r'\bFig\.\s([A-Z])'),
    re.compile(r'\bFig\.\s(\d)\((\w)\)'),
    re.compile(r'\bFig\.\s(\d)')
]

def process_passage(passage, section_type, base_filename):
    text = passage.get('text', '')
    # Replace the full stop after "Fig" followed by a digit with a space (e.g., "Fig. 4" -> "Fig 4")
    text = patterns[0].sub(r'Fig \g<0>', text)  # Use "g<0>" to capture the matched string
    # Replace "et al." with "et al" (e.g., "Hsua et al. showed" -> "Hsua et al showed")
    text = patterns[1].sub(r'et al ', text)
    # Replace "Fig." followed by a capital letter with "Fig" and the letter (e.g., "Fig. B" -> "Fig B")
    text = patterns[2].sub(r'Fig \1', text)
    # Replace "Fig." followed by a digit and a letter in parentheses with "Fig" and the digit and letter (e.g., "Fig. 4(c)" -> "Fig 4(c)")
    text = patterns[3].sub(r'Fig \1(\2)', text)
    # Replace "Fig." followed by a space and a digit (e.g., "Fig. 0" -> "Fig 0")
    text = patterns[4].sub(r'Fig \1', text)


    # Split the passage into sentences
    sentences = nltk.sent_tokenize(text)

    # Append each sentence as a new row in data_rows
    for sentence in sentences:
        data_rows.append({
            "filename": base_filename,
            "section_type": section_type,
            "sentence": sentence,
            "type": passage.get('infons', {}).get('type'),
        })

# Iterate over all JSON files in the directory
for filename in os.listdir(directory):
    if filename.endswith(".json"):
        logging.info(f"Processing file {filename}")
        try:
            with open(os.path.join(directory, filename), 'r') as file:
                data = json.load(file)
                base_filename = os.path.splitext(filename)[0]
                
                # Extract passages for each section type
                passages = data[0].get('documents', [{}])[0].get('passages', [])
                
                for section_type in section_types:
                    for passage in passages:
                        if (passage.get('infons', {}).get('section_type') == section_type and 
                                passage.get('infons', {}).get('type') != "title_1"):
                            process_passage(passage, section_type, base_filename)

        except json.JSONDecodeError as e:
            logging.error(f"Error parsing JSON file {filename}: {e}")
        except KeyError as e:
            logging.error(f"Error extracting text from JSON file {filename}: {e}")
        except Exception as e:
            logging.error(f"An unexpected error occurred while processing {filename}: {e}")

# Create a DataFrame from extracted rows if data_rows is not empty
if data_rows:
    df = pd.DataFrame(data_rows)
    logging.info(f"Extracted {len(df)} passages")
else:
    logging.info("No passages were extracted.")